<a href="https://colab.research.google.com/github/Soham-1009/GenAI-Practical/blob/main/Practical_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Configure device mapping for acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Module Implementations

### A. Sinusoidal Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a positional encoding matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # [max_len, 1]

        # Compute frequency scaling division term in log-space for numerical stability
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Fill even indices with sine and odd indices with cosine
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension -> [1, max_len, d_model]
        pe = pe.unsqueeze(0)

        # Register pe as a buffer so it resides in the state_dict but is not modified by gradients
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [batch_size, seq_len, d_model]
        seq_len = x.size(1)
        # Add positional representations directly to the token embeddings
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

### B. Custom Layer Normalization

In [ ]:
class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        # Learnable scale (gamma) and shift (beta) parameters
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute the mean and variance across the last dimension (embedding dimension)
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False) # standard population variance

        # Standardize the input
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        # Scale and shift
        return self.gamma * x_norm + self.beta

### C. Complete Simplified Transformer Encoder Block

In [ ]:
class SimplifiedTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # Multihead self-attention module
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        # Standard position-wise feedforward network
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        # Custom normalizations
        self.norm1 = CustomLayerNorm(d_model)
        self.norm2 = CustomLayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        # Sub-layer 1: Self-Attention + Residual Connection + Norm
        attn_output, _ = self.self_attn(src, src, src)
        src = src + self.dropout1(attn_output)
        src = self.norm1(src)

        # Sub-layer 2: Feed Forward network + Residual Connection + Norm
        ff_output = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(ff_output)
        src = self.norm2(src)

        return src

## 4. Visualizations & Numerical Comparisons
Let's visualize the sinusoidal pattern first, followed by a numerical verification comparing our custom LayerNorm implementation to PyTorch's native `nn.LayerNorm`.

In [ ]:
# 1. Visualizing Sinusoidal Positional Encoding
max_positions = 100
d_model_visualize = 128

pe_module = PositionalEncoding(d_model=d_model_visualize, max_len=max_positions, dropout=0.0)
pe_matrix = pe_module.pe.squeeze(0).cpu().numpy()

plt.figure(figsize=(12, 6))
plt.imshow(pe_matrix, cmap="RdBu", aspect="auto")
plt.colorbar(label="Amplitude")
plt.title("Sinusoidal Positional Encoding Map")
plt.xlabel("Embedding Dimension Index")
plt.ylabel("Token Position index")
plt.show()

In [ ]:
# 2. Parity Comparison between Custom Layer Normalization and Native PyTorch LayerNorm
x_test = torch.randn(2, 5, 256).to(device)

custom_ln = CustomLayerNorm(256).to(device)
pt_ln = nn.LayerNorm(256).to(device)

# Synchronize weight/bias states for strict numerical verification
with torch.no_grad():
    custom_ln.gamma.copy_(pt_ln.weight)
    custom_ln.beta.copy_(pt_ln.bias)

output_custom = custom_ln(x_test)
output_pt = pt_ln(x_test)

# Calculate structural discrepancy (Mean Absolute Difference)
difference = torch.abs(output_custom - output_pt).mean().item()
print(f"Mean Absolute Difference between Custom LayerNorm and PyTorch LayerNorm: {difference:.2e}")
assert difference < 1e-6, "Custom LayerNorm output differs significantly from PyTorch built-in!"

## 5. End-to-End System Integration Test
We will now run the model components on a batch of sequences.

In [ ]:
# End-to-End system demo
batch_size = 2
seq_len = 10
d_model = 512
vocab_size = 1000

token_embeddings = nn.Embedding(vocab_size, d_model).to(device)
positional_encoder = PositionalEncoding(d_model=d_model, max_len=50).to(device)
transformer_encoder = SimplifiedTransformerEncoderLayer(d_model=d_model, nhead=8, dim_feedforward=2048).to(device)

# Generate input token IDs
dummy_input = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
print(f"Input token IDs shape: {dummy_input.shape}")

# Pipeline pass
x = token_embeddings(dummy_input)
print(f"Token embeddings shape: {x.shape}")

x = positional_encoder(x)
print(f"Post positional encoding shape: {x.shape}")

output = transformer_encoder(x)
print(f"Transformer Output shape: {output.shape}")

# Assure there are no NaNs or Infs
assert not torch.isnan(output).any(), "Output contains NaN values!"
assert not torch.isinf(output).any(), "Output contains Inf values!"
print("Execution Successful! All assertions passed.")